In [10]:
import torch_geometric as tg
import torch 
import torch.nn as nn

from utils.data import build_data, get_target, get_neighbors
from utils.training import get_model, train

import ase.io
import pandas as pd
import numpy as np

In [11]:
df = pd.read_json('../database.json')
df.dropna(inplace=True)

structures = []
for index,row in df.iterrows():
    structures.append(ase.io.read(f'../structures/{row["index"]}.cif'))
df['structure'] = structures

In [12]:
df.head()

,index,comp,ID,dir_name,a2F_Freq_meV_sigma_5,a2F_sigma_5,a2F_Freq_meV_sigma_2,a2F_sigma_2,PhFreq_meV,Tot_PhDOS,...,ACH_w_sq_sigma_5,Ph_2x2x2_interpolated_Freq_meV,Ph_2x2x2_interpolated_Tot_DOS,Ph_2x2x2_interpolated_Site_Proj_DOS,AW,unitcell_vol,phlam,phwlog,phw2,structure
0,0,TiZr2Hf,agm001117008,batch-a/TiZr2Hf_agm001117008,"[0.30996106630000003, 0.43394440430000003, 0.5...","[3.7883000000000002e-06, 1.0395000000000001e-0...","[0.30996106630000003, 0.43394440430000003, 0.5...","[3.7410000000000003e-06, 1.02653e-05, 2.181740...","[-3.278e-07, 0.1239839295, 0.24796818680000002...","[0.0, 2.96332e-05, 0.0001185329, 0.0002666991,...",...,146.740253,"[-2.871e-07, 0.1000005794, 0.200001446, 0.3000...","[0.0, 3.59202e-05, 0.0001436812, 0.000323283, ...","[[0.0, 1.5672200000000002e-05, 6.26434e-05, 0....","[178.49, 91.224, 91.224, 47.867]",82.790253,2.025927,10.999947,13.204543,"(Atom('Hf', [1.55454, 1.55454, 4.282375], inde..."
1,1,Ti2Tc,agm001850181,batch-a/Ti2Tc_agm001850181,"[0.30995970570000003, 0.43394440430000003, 0.5...","[3.66e-06, 1.00429e-05, 2.4300300000000002e-05...","[0.30995970570000003, 0.43394440430000003, 0.5...","[4.5417e-06, 1.24625e-05, 2.97945e-05, 7.39756...","[-7.408000000000001e-07, 0.1239835165, 0.24796...","[0.0, 9.2863e-06, 3.71452e-05, 8.3576700000000...",...,440.894069,"[-6.861e-07, 0.10000018050000001, 0.2000010470...","[0.0, 1.7984e-06, 7.1937e-06, 1.61859e-05, 2.8...","[[0.0, 4.442e-07, 1.7758e-06, 3.99260000000000...","[47.867, 47.867, 97.90721]",45.683695,0.847788,19.976235,22.268911,"(Atom('Ti', [5.600596024987085, 1.746827397404..."
2,2,TiNb2Mo,agm002322068,batch-a/TiNb2Mo_agm002322068,"[0.30995970570000003, 0.43394440430000003, 0.5...","[1.0947e-05, 3.00387e-05, 6.38432e-05, 0.00027...","[0.30995970570000003, 0.43394440430000003, 0.5...","[1.24171e-05, 3.40725e-05, 7.241660000000001e-...","[-6.267000000000001e-07, 0.1239836307, 0.24796...","[0.0, 3.37434e-05, 0.0001349738, 0.00030369100...",...,348.385639,"[-3.821e-07, 0.1000004844, 0.20000135100000002...","[0.0, 1.7203300000000002e-05, 6.88133e-05, 0.0...","[[0.0, 2.5053e-06, 1.00497e-05, 2.26755e-05, 4...","[47.867, 92.90637, 92.90637, 95.95]",66.085084,1.394697,15.328235,18.122709,"(Atom('Ti', [0.0, 0.0, 0.0], index=0), Atom('N..."
3,3,Cr4TaW,agm003199689,batch-a/Cr4TaW_agm003199689,"[0.30995970570000003, 0.43394440430000003, 0.5...","[5.095000000000001e-07, 1.3981e-06, 2.9714e-06...","[0.30995970570000003, 0.43394440430000003, 0.5...","[6.890000000000001e-07, 1.8906000000000002e-06...","[-1.1286e-06, 0.1239831287, 0.247967386, 0.371...","[0.0, 1.18843e-05, 4.75373e-05, 0.000106958900...",...,484.472165,"[-1.1703e-06, 0.0999996962, 0.2000005628, 0.30...","[0.0, 1.23516e-05, 4.94068e-05, 0.0001111655, ...","[[0.0, 3.9042e-06, 1.56254e-05, 3.517620000000...","[180.94788, 51.9961, 51.9961, 51.9961, 51.9961...",78.413874,1.789248,17.902765,22.689554,"(Atom('Ta', [2.4021800000000004, 1.38689926964..."
4,4,ScZr2Hf,agm001224362,batch-a/ScZr2Hf_agm001224362,"[0.30995970570000003, 0.43394440430000003, 0.5...","[1.9878000000000003e-06, 5.4544e-06, 1.15927e-...","[0.30995970570000003, 0.43394440430000003, 0.5...","[2.71e-06, 7.4363e-06, 1.58048e-05, 3.85861e-0...","[-4.7410000000000004e-07, 0.1239837832, 0.2479...","[0.0, 2.55897e-05, 0.0001023588, 0.00023030740...",...,193.258820,"[-4.4220000000000003e-07, 0.1000004243, 0.2000...","[0.0, 1.77382e-05, 7.09527e-05, 0.0001596437, ...","[[0.0, 7.8426e-06, 3.15403e-05, 7.1347e-05, 0....","[178.49, 91.224, 91.224, 44.955908]",88.499474,1.700905,12.417268,14.220628,"(Atom('Hf', [2.23119, 2.23119, 2.22217], index..."


In [13]:
r_max = 4

df['Freq_meV'] = df['a2F_Freq_meV_sigma_2']
df['a2F'] = df['a2F_sigma_2']

df['target'] = df.apply(get_target,axis=1)
df['formula'] = df['structure'].map(lambda x: x.get_chemical_formula())
df['species'] = df['structure'].map(lambda x: list(set(x.get_chemical_symbols())))
species = sorted(list(set(df['species'].sum())))

df['data'] = df.apply(build_data,embed_ph_dos=False,embed_e_dos=False,fine=False,r_max = r_max,axis=1)


In [14]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

out_dim = len(df.iloc[0]['target'])
in_dim = len(df.iloc[0].data.x[0])
em_dim = 64

In [15]:
init_dict_base = dict(in_dim=118,                           
    em_dim=em_dim,                        
    irreps_in=str(em_dim)+"x0e",          
    irreps_out=str(out_dim)+"x0e",        
    irreps_node_attr=str(em_dim)+"x0e",   
    layers=2,                             
    mul=32,                               
    lmax=1,                               
    max_radius=r_max,      
    num_neighbors=get_neighbors(df,df.index).mean(),      
    reduce_output=True,
    p=0.0
)

In [16]:
max_iter = 10

In [17]:
class EMDLoss(nn.Module):
    """
    Earth Mover's Distance (EMD) loss with optional second derivative regularization.

    This loss function computes the Earth Mover's Distance (EMD) between two probability 
    distributions `p` and `q` using their cumulative distribution functions (CDFs). 
    An optional second derivative penalty can be applied to promote smoothness in `q`.

    Parameters:
    -----------
    second_derivative_weight : float, optional (default=10.0)
        Weight for the second derivative regularization term. Set to 0 to disable.

    Forward Inputs:
    ---------------
    p : torch.Tensor
        Target distribution of shape (batch_size, num_bins), expected to be normalized.
    q : torch.Tensor
        Predicted distribution of shape (batch_size, num_bins), expected to be normalized.

    Returns:
    --------
    total_loss : torch.Tensor
        Scalar loss value combining EMD and the second derivative regularization (if enabled).
    """
    
    def __init__(self, second_derivative_weight=10.0):
        super(EMDLoss, self).__init__()
        self.second_derivative_weight = second_derivative_weight

    def forward(self, p, q):
        cdf_p = torch.cumsum(p, dim=1)
        cdf_q = torch.cumsum(q, dim=1)

        emd = torch.sum(torch.abs(cdf_p - cdf_q), dim=1).mean()
        total_loss = emd

        if self.second_derivative_weight != 0:
            second_deriv_q = torch.sum(torch.abs(q[:, 2:] - 2 * q[:, 1:-1] + q[:, :-2]), dim=1).mean()
            total_loss = total_loss + self.second_derivative_weight *  second_deriv_q
        return total_loss


# Define loss function

In [18]:
loss_functions = ['EMD', 'MSE']
loss_function = 'EMD'

In [19]:
if loss_function == 'MSE':
    loss_fn = torch.nn.MSELoss()
    loss_fn_mae = torch.nn.MSELoss()
elif loss_function == 'EMD':
    loss_fn = EMDLoss()
    loss_fn_mae = EMDLoss()

In [20]:
folds = range(2)
for i in folds:

    idx_train = pd.Index(np.loadtxt(f'indices/idx_train__cpd_{i}.txt'))
    idx_valid = pd.Index(np.loadtxt(f'indices/idx_val__cpd_{i}.txt'))
    
    dataloader_train = tg.loader.DataLoader(df.loc[idx_train]['data'].values, batch_size=512)
    dataloader_valid = tg.loader.DataLoader(df.iloc[df.index.isin(idx_valid)]['data'].values, batch_size=512)

    name = f'../models/CSO_test_{loss_function}_{i}'
    
    run_name = f'{name}'
    print(name)

    model, opt, scheduler = get_model(init_dict_base,lr = 0.005,wd=0,device=device)

    train(model, opt, dataloader_train, dataloader_valid,
          loss_fn, loss_fn_mae, run_name,
          max_iter=max_iter, scheduler=scheduler, device=device)

FileNotFoundError: indices/idx_train__cpd_0.txt not found.